---
# Phần B — Mô hình VGG16 Conv + Transformer Decoder (Mở rộng)

Phần này xây dựng kiến trúc thay thế, dùng Transformer Decoder thay vì LSTM. Hai thay đổi chính so với Phần A:

**1. Biểu diễn ảnh:** Thay vector phẳng (4096-D) bằng chuỗi 49 patch vector (49, 512) — giữ thông tin vị trí không gian.

**2. Decoder:** Thay LSTM + Add bằng Transformer Decoder với cross-attention — mỗi từ được sinh có thể "nhìn" vào các vùng ảnh khác nhau.

```
Encoder:                                          Decoder:
                                                  ┌─────────────────────────────────────┐
VGG16 block5_pool                                 │ Input: caption tokens (max_length)   │
  (7×7×512)                                       │   ↓                                  │
     ↓                                            │ Embedding(d_model=256)               │
reshape (49, 512)                                 │   ↓                                  │
     ↓                                            │ + Positional Encoding                │
Dense(256, relu) ──→ enc_proj (49, 256)  ──────→  │   ↓                                  │
                                         cross    │ Masked Self-Attention                │
                                         attn     │   ↓                                  │
                                          ↗       │ Cross-Attention (Q=text, KV=image)   │
                                                  │   ↓                                  │
                                                  │ Feed-Forward Network                 │
                                                  │   ↓ (×N lớp)                         │
                                                  │ output[:, -1, :] → Dense(V, softmax) │
                                                  └─────────────────────────────────────┘
```


## B.1. Tạo Cặp Input-Output cho Transformer

Tương tự LSTM, mỗi mô tả sinh (T-1) cặp training. Khác biệt duy nhất:

- **X1**: `features_conv[key]` có shape (49, 512) thay vì (1, 4096)
- Không cần `[0]` vì features_conv đã reshape thành (49, 512) khi trích xuất

Khi stack thành batch: X1 có shape (N, 49, 512) — mỗi sample là chuỗi 49 patch.

In [ ]:
def create_sequences_transformer(tokenizer, max_length, descriptions, photos, vocab_size):
    """Tạo cặp input-output cho mô hình Transformer.

    Khác LSTM: X1 có shape (49, 512) thay vì (4096,).

    Trả về:
        X1: np.array (N, 49, 512) — chuỗi patch ảnh
        X2: np.array (N, max_length) — chuỗi từ đã padding
        y:  np.array (N, vocab_size) — từ tiếp theo, one-hot
    """
    X1, X2, y = list(), list(), list()
    for key, desc_list in descriptions.items():
        if key not in photos:
            continue
        for desc in desc_list:
            seq = tokenizer.texts_to_sequences([desc])[0]
            for i in range(1, len(seq)):
                in_seq, out_seq = seq[:i], seq[i]
                in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                X1.append(photos[key])  # (49, 512) — không cần [0]
                X2.append(in_seq)
                y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

print('Tạo dữ liệu cho Transformer (tập train)...')
X1train_t, X2train_t, ytrain_t = create_sequences_transformer(
    tokenizer, max_length, train_descriptions, train_features_conv, vocab_size
)
print(f'  X1train_t (patch ảnh): {X1train_t.shape}')
print(f'  X2train_t (chuỗi): {X2train_t.shape}')
print(f'  ytrain_t (nhãn): {ytrain_t.shape}')

print('\nTạo dữ liệu cho Transformer (tập dev)...')
X1dev_t, X2dev_t, ydev_t = create_sequences_transformer(
    tokenizer, max_length, dev_descriptions, dev_features_conv, vocab_size
)
print(f'  X1dev_t: {X1dev_t.shape}')

## B.2. Định nghĩa Mô hình Transformer Decoder

### Ba thành phần custom

**1. PositionalEncoding:** Cộng thông tin vị trí vào token embedding. Dùng sinusoidal encoding cố định (không học):

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Tần số giảm dần theo chiều $i$: các chiều đầu mã hoá vị trí ở tần số cao (phân biệt vị trí gần nhau), các chiều sau mã hoá ở tần số thấp (phân biệt vị trí xa nhau).

**2. TransformerDecoderLayer:** Một lớp decoder gồm 3 sub-layer:
- `self_attn`: Masked MHSA — kết hợp causal mask (chặn tương lai) và padding mask (chặn vị trí padding = 0). `use_causal_mask=True` tạo mask tam giác, `attention_mask` nhận padding mask từ bên ngoài. Hai mask được kết hợp tự động bởi Keras MHA.
- `cross_attn`: MHSA với Query từ text, Key và Value từ encoder ảnh (49 patch). Cho phép mỗi từ tập trung vào vùng ảnh liên quan.
- `ffn`: 2 lớp Dense (dff → d_model). Biến đổi phi tuyến sau attention.

Mỗi sub-layer: output = LayerNorm(input + Dropout(SubLayer(input)))

**3. define_model_transformer:** Ghép nối toàn bộ.

### Hyperparameter

| Tham số | Giá trị | Lý do |
|---------|---------|-------|
| d_model | 256 | Cùng chiều với LSTM để so sánh công bằng |
| num_heads | 4 | 4 head × 64 key_dim = 256 |
| dff | 512 | 2× d_model theo quy ước |
| num_layers | 2 | Flickr8k nhỏ, 6 lớp sẽ overfit |
| dropout | 0.1 | Thấp hơn LSTM (0.5) vì Transformer có nhiều cơ chế regularization khác |
| learning_rate | 1e-4 | Nhỏ hơn mặc định Adam (1e-3) vì Transformer nhạy với lr |

### Tại sao mask_zero=False cho Transformer?

Khác với LSTM, Transformer dùng `mask_zero=False` vì hai lý do kỹ thuật:

**(1) Vấn đề tương thích MultiHeadAttention:** MultiHeadAttention trong TF2/Keras khi kết hợp với `use_causal_mask=True` không tương thích với mask propagation từ Embedding — gây lỗi shape hoặc bỏ qua causal mask.

**(2) Padding ở bên trái:** Padding nằm bên TRÁI chuỗi (left-padding, do `pad_sequences` mặc định `padding='pre'`). Causal mask cho phép vị trí cuối attend toàn bộ chuỗi — bao gồm padding bên trái. Embedding(index=0) khởi tạo gần zero → đóng góp attention thấp. **Hệ quả:** mô hình hoạt động đúng mà không cần explicit padding mask.

### Tại sao lấy `dec_output[:, -1, :]`?

Sau N lớp Transformer Decoder, ta chỉ cần output ở **vị trí cuối cùng** vì:
- Causal mask đảm bảo vị trí cuối đã tích luỹ thông tin từ **toàn bộ** chuỗi trước đó
- Cross-attention tại vị trí cuối đã "nhìn" toàn bộ 49 patch ảnh
- Tương tự cách LSTM trả về hidden state cuối cùng

Vị trí cuối chính là vị trí dự đoán từ tiếp theo: nếu chuỗi đầu vào là `"startseq little girl"`, vị trí cuối (tại "girl") sẽ dự đoán từ tiếp theo "running".

### Lưu ý về Learning Rate

**Learning rate 1e-4 cho Transformer nhỏ hơn 10× so với LSTM (1e-3).** Đây là lựa chọn **có chủ đích**: Transformer với attention mechanism có gradient landscape phức tạp hơn, lr lớn dễ gây divergence hoặc dao động val_loss. Vaswani et al. (2017) dùng warmup + decay schedule, nhưng trên Flickr8k nhỏ, lr cố định 1e-4 đủ ổn định.

In [ ]:
class PositionalEncoding(tf.keras.layers.Layer):
    """Positional Encoding cộng thông tin vị trí vào embedding.

    Dùng sinusoidal encoding cố định (không học).
    Tính toán ma trận PE một lần trong __init__ và lưu dưới dạng constant.
    """
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model
        # Tính ma trận positional encoding: shape (1, max_len, d_model)
        positions = np.arange(max_len)[:, np.newaxis]      # (max_len, 1)
        dims = np.arange(d_model)[np.newaxis, :]            # (1, d_model)
        angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])  # Chiều chẵn: sin
        angles[:, 1::2] = np.cos(angles[:, 1::2])  # Chiều lẻ: cos
        self.pos_encoding = tf.constant(angles[np.newaxis, :, :], dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        # Chỉ lấy PE tương ứng với độ dài chuỗi thực tế
        return x + self.pos_encoding[:, :seq_len, :]

    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len, 'd_model': self.d_model})
        return config


class TransformerDecoderLayer(tf.keras.layers.Layer):
    """Một lớp Transformer Decoder.

    Gồm 3 sub-layer, mỗi sub-layer có residual connection + LayerNorm:
    1. Masked Multi-Head Self-Attention (causal mask + padding mask)
    2. Multi-Head Cross-Attention (attend vào encoder output, có padding mask)
    3. Position-wise Feed-Forward Network
    """
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.dff = dff
        self.dropout_rate = dropout_rate

        # key_dim = d_model / num_heads để tổng chiều = d_model
        self.self_attn = MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.cross_attn = MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(dff, activation='relu'),    # Mở rộng lên dff chiều
            Dense(d_model)                     # Chiếu lại về d_model
        ])
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()
        self.norm3 = LayerNormalization()
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
        self.dropout3 = Dropout(dropout_rate)

    def call(self, x, encoder_output, padding_mask=None, training=False):
        # Sub-layer 1: Masked Self-Attention
        # use_causal_mask=True tạo mask tam giác — vị trí i chỉ attend vị trí ≤ i
        # attention_mask chặn thêm các vị trí padding (giá trị 0 trong chuỗi gốc)
        attn1 = self.self_attn(
            query=x, value=x, key=x,
            attention_mask=padding_mask,
            use_causal_mask=True, training=training
        )
        x = self.norm1(x + self.dropout1(attn1, training=training))

        # Sub-layer 2: Cross-Attention
        # Query = text embeddings, Key = Value = encoder output (patch ảnh)
        # Không cần padding mask cho encoder (49 patch luôn đầy đủ, không padding)
        attn2 = self.cross_attn(
            query=x, value=encoder_output, key=encoder_output,
            training=training
        )
        x = self.norm2(x + self.dropout2(attn2, training=training))

        # Sub-layer 3: Feed-Forward Network
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_out, training=training))
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self.d_model, 'num_heads': self.num_heads,
            'dff': self.dff, 'dropout_rate': self.dropout_rate
        })
        return config


def define_model_transformer(vocab_size, max_length, patch_dim=512,
                              num_patches=49, d_model=256, num_heads=4,
                              dff=512, num_layers=2, dropout_rate=0.1):
    """Mô hình Image Captioning dựa trên Transformer Decoder.

    Tham số:
        vocab_size: kích thước từ vựng (bao gồm padding index 0)
        max_length: độ dài mô tả tối đa (xác định kích thước PE và padding)
        patch_dim: chiều mỗi patch ảnh (512 từ VGG16 block5_pool)
        num_patches: số patch (7×7 = 49)
        d_model: chiều embedding = chiều hidden state (256)
        num_heads: số attention head (4)
        dff: chiều ẩn trong FFN (512)
        num_layers: số lớp Transformer Decoder (2)
        dropout_rate: tỷ lệ dropout (0.1)
    """
    # --- Encoder ảnh ---
    image_input = Input(shape=(num_patches, patch_dim), name='image_patches')
    # Linear projection: 512 → d_model (256)
    enc_proj = Dense(d_model, activation='relu')(image_input)
    enc_proj = Dropout(dropout_rate)(enc_proj)
    # enc_proj: (batch, 49, 256)

    # --- Decoder chuỗi ---
    seq_input = Input(shape=(max_length,), name='sequence_input')

    # Tạo padding mask: vị trí nào trong seq_input == 0 sẽ bị mask
    # Shape: (batch, 1, 1, max_length) — broadcast cho multi-head attention
    padding_mask = tf.cast(tf.math.not_equal(seq_input, 0), dtype=tf.float32)
    # Reshape thành (batch, 1, 1, max_length) để MHA broadcast đúng
    padding_mask = padding_mask[:, tf.newaxis, tf.newaxis, :]

    # Token Embedding: index → vector d_model chiều
    # mask_zero=False vì ta tự quản lý padding mask qua attention_mask
    token_emb = Embedding(vocab_size, d_model, mask_zero=False)(seq_input)
    # Positional Encoding: cộng thông tin vị trí
    token_emb = PositionalEncoding(max_length, d_model)(token_emb)
    token_emb = Dropout(dropout_rate)(token_emb)
    # token_emb: (batch, max_length, 256)

    # Stack N lớp Transformer Decoder
    dec_output = token_emb
    for i in range(num_layers):
        dec_output = TransformerDecoderLayer(
            d_model, num_heads, dff, dropout_rate,
            name=f'decoder_layer_{i}'
        )(dec_output, enc_proj, padding_mask=padding_mask)
    # dec_output: (batch, max_length, 256)

    # Lấy output ở vị trí cuối cùng — vị trí dự đoán từ tiếp theo
    dec_output = dec_output[:, -1, :]  # (batch, 256)

    # Lớp phân loại: Dense(256) → Dense(vocab_size, softmax)
    dec_output = Dense(d_model, activation='relu')(dec_output)
    dec_output = Dropout(dropout_rate)(dec_output)
    outputs = Dense(vocab_size, activation='softmax')(dec_output)

    model = Model(inputs=[image_input, seq_input], outputs=outputs)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4)
    )
    return model

model_transformer = define_model_transformer(vocab_size, max_length)
model_transformer.summary()


## B.3. Huấn luyện Mô hình Transformer

Cấu hình tương tự LSTM nhưng:
- **Epoch**: 30 (nhiều hơn vì lr nhỏ hơn, hội tụ chậm hơn)
- **Learning rate**: 1e-4 (nhỏ hơn 10× so với Adam mặc định)
- Thời gian: ~2-3 phút/epoch trên GPU T4 (chậm hơn LSTM do attention có complexity $O(n^2)$)

In [ ]:
transformer_model_path = os.path.join(WORK_DIR, 'model_transformer.h5')

if not os.path.exists(transformer_model_path):
    checkpoint_t = ModelCheckpoint(
        transformer_model_path, monitor='val_loss',
        verbose=1, save_best_only=True, mode='min'
    )
    early_stop_t = EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    print('Bắt đầu huấn luyện mô hình Transformer...')
    history_transformer = model_transformer.fit(
        [X1train_t, X2train_t], ytrain_t,
        epochs=30, verbose=2, batch_size=64,
        callbacks=[checkpoint_t, early_stop_t],
        validation_data=([X1dev_t, X2dev_t], ydev_t)
    )
    print('Huấn luyện xong.')
else:
    print(f'Mô hình Transformer đã có tại {transformer_model_path}')
    model_transformer = load_model(
        transformer_model_path,
        custom_objects={
            'PositionalEncoding': PositionalEncoding,
            'TransformerDecoderLayer': TransformerDecoderLayer
        }
    )

## B.4. Đánh giá Mô hình Transformer

Dùng cùng hàm `evaluate_model` đã định nghĩa ở Phần A, với `model_type='transformer'`. Hàm `generate_desc` đã xử lý reshape shape (49, 512) → (1, 49, 512) cho Transformer tự động.

In [ ]:
# Tải mô hình tốt nhất
model_transformer_best = load_model(
    transformer_model_path,
    custom_objects={
        'PositionalEncoding': PositionalEncoding,
        'TransformerDecoderLayer': TransformerDecoderLayer
    }
)

print('=== Đánh giá mô hình Transformer trên tập test ===')
bleu_transformer = evaluate_model(
    model_transformer_best, test_descriptions, test_features_conv,
    tokenizer, max_length, model_type='transformer'
)